# Standardize pollinator dataset to Camtrap DP (Insect AI extension)

Converts the filtered camera trap data (images in `media/`, Pascal VOC XMLs in `raw-data/`) into Camtrap DP tables.

Prerequisite: run `filter_images.ipynb` in the project root first to populate `media/` and `raw-data/`.

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image
from PIL.ExifTags import TAGS

raw_data = Path("../raw-data")
media_dir = Path("../media")
output_dir = Path("..")

def get_exif(img_path):
    """Return EXIF as {tag_name: value} dict, or empty dict."""
    try:
        exif = Image.open(img_path)._getexif()
        if not exif:
            return {}
        return {TAGS.get(k, k): v for k, v in exif.items()}
    except Exception:
        return {}

# Pair each XML with its image in media/, extract camera model and folder
target_images = []
for xml_file in sorted(raw_data.glob("*.xml")):
    stem = xml_file.stem
    for ext in [".jpg", ".JPG"]:
        img = media_dir / f"{stem}{ext}"
        if img.exists():
            exif = get_exif(img)
            tree = ET.parse(xml_file)
            folder = tree.getroot().find("folder").text
            target_images.append((stem, img, xml_file, exif, folder))
            break

print(f"Camera trap images: {len(target_images)}")

## Step 1: Build deployments table

Deployments are inferred from unique combinations of camera model (EXIF `Model`) and date. Each day of imaging is treated as a separate deployment, since the camera may have been repositioned between days.

In [ ]:
import pandas as pd
import json
from datetime import datetime
from zoneinfo import ZoneInfo

TZ = ZoneInfo("Europe/Madrid")

# Build a lookup: (camera_model, date) -> deploymentID
dep_timestamps = {}
media_to_dep = {}

for stem, img_path, xml_path, exif, folder in target_images:
    model = exif.get("Model", "unknown")

    dt_str = exif.get("DateTimeOriginal", "")
    if dt_str:
        dt_naive = datetime.strptime(dt_str, "%Y:%m:%d %H:%M:%S")
        dt_local = dt_naive.replace(tzinfo=TZ)
        date_key = dt_naive.date()
    else:
        dt_local = None
        date_key = None

    dep_key = (model, date_key)
    dep_timestamps.setdefault(dep_key, []).append(dt_local)
    media_to_dep[stem] = dep_key

# Create deployment IDs and rows
dep_key_to_id = {}
dep_rows = []
for i, dep_key in enumerate(sorted(dep_timestamps.keys()), 1):
    model, date_key = dep_key
    dep_id = f"dep{i:02d}"
    dep_key_to_id[dep_key] = dep_id

    ts_valid = [t for t in dep_timestamps[dep_key] if t is not None]
    dep_start = min(ts_valid).isoformat() if ts_valid else None
    dep_end = max(ts_valid).isoformat() if ts_valid else None

    dep_rows.append({
        "deploymentID": dep_id,
        "locationID": None,
        "locationName": None,
        "latitude": 39.6,
        "longitude": 2.9,
        "coordinateUncertainty": 100000,
        "deploymentStart": dep_start,
        "deploymentEnd": dep_end,
        "setupBy": "Pau Enric Serra Marin",
        "cameraID": model,
        "cameraModel": f"RaspberryPi-{model}",
        "cameraDelay": None,
        "cameraHeight": None,
        "cameraDepth": None,
        "cameraTilt": None,
        "cameraHeading": None,
        "detectionDistance": None,
        "timestampIssues": False,
        "baitUse": False,
        "featureType": None,
        "habitat": None,
        "deploymentGroups": None,
        "deploymentTags": None,
        "deploymentComments": "Approximate coordinates (Balearic Islands). Exact locations awaiting data from controller.",
    })

deployments = pd.DataFrame(dep_rows)
print(f"{len(deployments)} deployments")
deployments[["deploymentID", "cameraID", "deploymentStart", "deploymentEnd"]]

## Step 2: Build media table

Build `media.csv` according to the [Camtrap DP media spec](https://camtrap-dp.tdwg.org/data/#media). Each media record is linked to its deployment via `deploymentID`.

Timestamps come from EXIF `DateTimeOriginal`, localized to Europe/Madrid.

In [ ]:
media_rows = []
for stem, img_path, xml_path, exif, folder in target_images:
    model = exif.get("Model", "unknown")
    dt_str = exif.get("DateTimeOriginal", "")
    date_key = datetime.strptime(dt_str, "%Y:%m:%d %H:%M:%S").date() if dt_str else None
    dep_id = dep_key_to_id[(model, date_key)]

    if dt_str:
        dt_naive = datetime.strptime(dt_str, "%Y:%m:%d %H:%M:%S")
        timestamp = dt_naive.replace(tzinfo=TZ).isoformat()
    else:
        timestamp = None

    exif_json = json.dumps({
        k: v for k, v in exif.items()
        if isinstance(v, (str, int, float, bool))
    })

    media_rows.append({
        "mediaID": stem,
        "deploymentID": dep_id,
        "captureMethod": "timeLapse",
        "timestamp": timestamp,
        "filePath": f"media/{img_path.name}",
        "filePublic": True,
        "fileName": img_path.name,
        "fileMediatype": "image/jpeg",
        "exifData": exif_json,
        "favorite": None,
        "mediaComments": None,
    })

media = pd.DataFrame(media_rows)
print(f"{len(media)} media records")
print(f"Timestamp range: {media['timestamp'].min()} to {media['timestamp'].max()}")
print(f"\nPer deployment:")
print(media.groupby("deploymentID").size())
media.head()

In [ ]:
media.to_csv(output_dir / "media.csv", index=False)
print("Saved media.csv")

## Step 3: Build observations table

Build `observations.csv` from the Pascal VOC XML annotations. Each `<object>` becomes one observation row, linked to both `mediaID` and `deploymentID`. Bounding boxes are normalized to 0–1 range.

In [ ]:
obs_rows = []
obs_counter = 0

for stem, img_path, xml_path, exif, folder in target_images:
    model = exif.get("Model", "unknown")
    dt_str = exif.get("DateTimeOriginal", "")
    date_key = datetime.strptime(dt_str, "%Y:%m:%d %H:%M:%S").date() if dt_str else None
    dep_id = dep_key_to_id[(model, date_key)]

    tree = ET.parse(xml_path)
    root = tree.getroot()

    img_w = int(root.find(".//size/width").text)
    img_h = int(root.find(".//size/height").text)

    media_row = media.loc[media["mediaID"] == stem].iloc[0]
    timestamp = media_row["timestamp"]

    for obj in root.findall("object"):
        obs_counter += 1
        name = obj.find("name").text

        bb = obj.find("bndbox")
        xmin = int(bb.find("xmin").text)
        ymin = int(bb.find("ymin").text)
        xmax = int(bb.find("xmax").text)
        ymax = int(bb.find("ymax").text)

        obs_rows.append({
            "observationID": f"obs{obs_counter:04d}",
            "deploymentID": dep_id,
            "mediaID": stem,
            "eventID": None,
            "eventStart": timestamp,
            "eventEnd": timestamp,
            "observationLevel": "media",
            "observationType": "animal",
            "cameraSetupType": None,
            "scientificName": name,
            "count": 1,
            "lifeStage": None,
            "sex": None,
            "behavior": None,
            "individualID": None,
            "individualPositionRadius": None,
            "individualPositionAngle": None,
            "individualSpeed": None,
            "bboxX": round(xmin / img_w, 6),
            "bboxY": round(ymin / img_h, 6),
            "bboxWidth": round((xmax - xmin) / img_w, 6),
            "bboxHeight": round((ymax - ymin) / img_h, 6),
            "classificationMethod": "human",
            "classifiedBy": "Pau Enric Serra Marin",
            "classificationTimestamp": None,
            "classificationProbability": None,
            "observationTags": None,
            "observationComments": None,
        })

observations = pd.DataFrame(obs_rows)
print(f"{len(observations)} observations from {observations['mediaID'].nunique()} media files")
print(f"\nPer deployment:")
print(observations.groupby("deploymentID").size())
print(f"\nTop taxa:")
print(observations["scientificName"].value_counts().head(10))

## Step 3b: Build detections table (InsectAI extension)

The detections table captures intermediate human and machine classifications, as defined by the [InsectAI Camtrap DP extension](REPORT_2-InsectAI-CamtrapDP_Extension_v1_0.pdf). In this dataset there is a single source of classification (human expert Pau Enric Serra Marin), so detections are identical to observations. No `modelID` is needed since `classificationMethod` is `human`.

In [ ]:
# Build detections from observations, renaming fields per the InsectAI extension spec
detections = observations.rename(columns={
    "observationID": "detectionID",
    "observationLevel": "detectionLevel",
    "observationType": "detectionType",
    "observationTags": "detectionTags",
    "observationComments": "detectionComments",
}).copy()

# Renumber IDs as det*
detections["detectionID"] = [f"det{i:04d}" for i in range(1, len(detections) + 1)]

# Add extension-specific fields
detections["sourceDetectionID"] = None  # no prior detection to reference
detections["modelID"] = None  # human classification, no model

print(f"{len(detections)} detections")
detections.head()

In [ ]:
observations.to_csv(output_dir / "observations.csv", index=False)
detections.to_csv(output_dir / "detections.csv", index=False)
deployments.to_csv(output_dir / "deployments.csv", index=False)
print("Saved observations.csv")
print("Saved detections.csv")
print("Saved deployments.csv")

## Step 4: Build datapackage.json

Package metadata following the [Camtrap DP spec](https://camtrap-dp.tdwg.org/metadata/). Taxonomic list and temporal extent are derived from the data.

In [ ]:
from datetime import timezone

# Derive temporal extent from media timestamps
timestamps = pd.to_datetime(media["timestamp"], utc=True)
temporal_start = timestamps.min().strftime("%Y-%m-%d")
temporal_end = timestamps.max().strftime("%Y-%m-%d")

# Derive unique taxa from observations
taxa = [
    {"scientificName": name}
    for name in sorted(observations["scientificName"].unique())
]

datapackage = {
    "resources": [
        {
            "name": "deployments",
            "path": "deployments.csv",
            "profile": "tabular-data-resource",
            "format": "csv",
            "mediatype": "text/csv",
            "encoding": "utf-8",
            "schema": "https://raw.githubusercontent.com/tdwg/camtrap-dp/1.0.2/deployments-table-schema.json",
        },
        {
            "name": "media",
            "path": "media.csv",
            "profile": "tabular-data-resource",
            "format": "csv",
            "mediatype": "text/csv",
            "encoding": "utf-8",
            "schema": "https://raw.githubusercontent.com/tdwg/camtrap-dp/1.0.2/media-table-schema.json",
        },
        {
            "name": "observations",
            "path": "observations.csv",
            "profile": "tabular-data-resource",
            "format": "csv",
            "mediatype": "text/csv",
            "encoding": "utf-8",
            "schema": "https://raw.githubusercontent.com/tdwg/camtrap-dp/1.0.2/observations-table-schema.json",
        },
        {
            "name": "detections",
            "path": "detections.csv",
            "profile": "tabular-data-resource",
            "format": "csv",
            "mediatype": "text/csv",
            "encoding": "utf-8",
        },
    ],
    "profile": "https://raw.githubusercontent.com/tdwg/camtrap-dp/1.0.2/camtrap-dp-profile.json",
    "name": "plant-pollinator-interactions-balearic-islands",
    "created": datetime.now(timezone.utc).isoformat(),
    "title": "Plant-pollinator interactions in the Balearic Islands",
    "description": "Camera trap images of pollinators visiting flowers, collected as part of a citizen science project in the Balearic Islands (Mediterranean ecosystem, Spain). Images were captured using Raspberry Pi cameras during active pollination events in natural field conditions.",
    "contributors": [
        {
            "title": "Pau Enric Serra Marin",
            "role": "contact",
        }
    ],
    "project": {
        "title": "Plant-pollinator interactions in the Balearic Islands",
        "captureMethod": ["timeLapse"],
        "individualAnimals": False,
        "observationLevel": ["media"],
    },
    "temporal": {
        "start": temporal_start,
        "end": temporal_end,
    },
    "taxonomic": taxa,
}

with open(output_dir / "datapackage.json", "w") as f:
    json.dump(datapackage, f, indent=2)

print("Saved datapackage.json")
print(f"Temporal: {temporal_start} to {temporal_end}")
print(f"Taxa: {len(taxa)}")